# Size scaling — point cloud

Write/read/disk-size of point clouds at increasing `N`, with **CSV as a
baseline** for context. Same `chunk_shape` across runs so the only
variable is vertex count.

Each timing is averaged over **`N_RUNS = 10` runs**; the plot shows the
mean with a shaded **95% confidence interval** (Student's t, df=9).

For each `N` we measure:

| Operation | zarr-vectors | CSV (baseline) |
| --- | --- | --- |
| Write     | `write_points`           | `pandas.to_csv` |
| Read all  | `read_points`            | `pandas.read_csv` |
| Read one  | one chunk via lazy API   | `read_csv` at a **random row index** |
| Disk size | store directory          | CSV file |

CSV "read one" uses a random row each run (CSV has no random access, so the
parser must scan from the file head — this is the honest comparison vs.
zarr-vectors' chunk-level lookup).

Runtime: ~5 minutes on a laptop (the 1M case dominates).

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.lazy import open_zv

SIZES = [1_000, 10_000, 100_000, 1_000_000]
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0


def _csv_path(prefix):
    """Fresh tempdir + CSV path."""
    return Path(tempfile.mkdtemp(prefix=f'csvbench_{prefix}_')) / 'points.csv'


def _csv_write(path, positions, intensity):
    """Baseline: write x,y,z,intensity columns to a CSV."""
    pd.DataFrame({
        'x': positions[:, 0],
        'y': positions[:, 1],
        'z': positions[:, 2],
        'intensity': intensity,
    }).to_csv(path, index=False)


def _csv_read_all(path):
    """Read every row back into memory."""
    return pd.read_csv(path)


def _csv_read_one(path, row_idx):
    """Random-index single-row read. CSV has no random access, so the
    parser must scan from the top — this exposes the linear-scan cost."""
    return pd.read_csv(path, skiprows=range(1, row_idx + 1), nrows=1)


def _zv_read_one(store_path):
    """Read just one chunk's worth of vertices via the lazy API."""
    zv = open_zv(store_path)
    chunk_keys = zv[0].vertices._chunk_keys  # noqa: SLF001 — minimal demo
    if not chunk_keys:
        return None
    return zv[0].vertices[chunk_keys[0]].compute()

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)

# Per-(metric, prefix) raw timings; shape becomes (len(SIZES), N_RUNS) after fill.
metrics = ['write_s', 'read_all_s', 'read_one_s']
raw = {m: {p: np.zeros((len(SIZES), N_RUNS)) for p in ('zv', 'csv')} for m in metrics}
sizes_MB = {'zv': np.zeros(len(SIZES)), 'csv': np.zeros(len(SIZES))}

for i, n in enumerate(SIZES):
    # Generate input data once per N (re-running write_points with the same
    # input is the realistic re-run scenario; we still get fresh stores).
    positions = rng.uniform(0, 1000, (n, 3)).astype(np.float32)
    intensity = rng.uniform(0, 1, n).astype(np.float32)

    for run in range(N_RUNS):
        # ---- ZV: fresh store each run (cold-cache write) ----
        store = _new_store(f'size_{n}_r{run}')
        t_zv_write,    _ = _time(
            write_points, store, positions,
            chunk_shape=CHUNK, bin_shape=BIN,
            vertex_attributes={'intensity': intensity},
        )
        t_zv_read_all, _ = _time(read_points, store, attribute_names=['intensity'])
        t_zv_read_one, _ = _time(_zv_read_one, store)
        if run == 0:
            sizes_MB['zv'][i] = _store_bytes(store) / 1e6
        shutil.rmtree(Path(store).parent, ignore_errors=True)

        # ---- CSV baseline ----
        csv = _csv_path(f'size_{n}_r{run}')
        t_csv_write,    _ = _time(_csv_write, csv, positions, intensity)
        t_csv_read_all, _ = _time(_csv_read_all, csv)
        # Fresh random row index per run — CSV pays linear scan cost.
        row_idx = int(rng.integers(0, n))
        t_csv_read_one, _ = _time(_csv_read_one, csv, row_idx)
        if run == 0:
            sizes_MB['csv'][i] = csv.stat().st_size / 1e6
        shutil.rmtree(csv.parent, ignore_errors=True)

        raw['write_s']['zv'][i, run]     = t_zv_write
        raw['read_all_s']['zv'][i, run]  = t_zv_read_all
        raw['read_one_s']['zv'][i, run]  = t_zv_read_one
        raw['write_s']['csv'][i, run]    = t_csv_write
        raw['read_all_s']['csv'][i, run] = t_csv_read_all
        raw['read_one_s']['csv'][i, run] = t_csv_read_one

# Summarise into a tidy dataframe (mean + 95% CI half-width per metric).
rows = []
for i, n in enumerate(SIZES):
    row = {'N': n}
    for m in metrics:
        for p in ('zv', 'csv'):
            mean, hw = _mean_ci95(raw[m][p][i])
            row[f'{p}_{m}_mean'] = round(mean, 4)
            row[f'{p}_{m}_hw']   = round(hw, 4)
    row['zv_size_MB']  = round(sizes_MB['zv'][i],  2)
    row['csv_size_MB'] = round(sizes_MB['csv'][i], 2)
    rows.append(row)

df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
PANELS = [
    ('Write time', 'write_s',    's'),
    ('Read all',   'read_all_s', 's'),
    ('Read one',   'read_one_s', 's'),
]
SERIES = [
    ('zarr-vectors', 'zv',  'C0'),
    ('csv',          'csv', 'C1'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Three timing panels: mean line + shaded 95% CI band on log-log.
for ax, (title, key, unit) in zip(axes[:3], PANELS):
    for label, prefix, color in SERIES:
        mean = df[f'{prefix}_{key}_mean'].values
        hw   = df[f'{prefix}_{key}_hw'].values
        ax.fill_between(df['N'], mean - hw, mean + hw, color=color, alpha=0.2)
        ax.loglog(df['N'], mean, 'o-', color=color, label=label)
    ax.set_xlabel('N (vertices)')
    ax.set_ylabel(unit)
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)

# Disk size panel: single deterministic measurement per series — no CI.
ax = axes[3]
for label, prefix, color in SERIES:
    ax.loglog(df['N'], df[f'{prefix}_size_MB'], 'o-', color=color, label=label)
ax.set_xlabel('N (vertices)')
ax.set_ylabel('MB')
ax.set_title('Disk size')
ax.grid(True, which='both', alpha=0.3)

axes[0].legend(loc='best')
fig.suptitle(
    f'zarr-vectors vs CSV — point cloud scaling ({N_RUNS} runs, 95% CI)',
)
plt.tight_layout()